<a href="https://colab.research.google.com/github/Sunauraa/Phishing-Scam_Detection/blob/main/blt_scam_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install datasets transformers scikit-learn pandas pyarrow

In [2]:
import math, random, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

warnings.filterwarnings("ignore", message=".*nested tensors.*")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

CFG = dict(
    seed=42,
    max_bytes=256,          # truncate every text to this many UTF-8 bytes
    max_bpe_tokens=128,     # truncate for the BPE baseline
    # entropy model (stage 1)
    ent_d=128, ent_layers=2, ent_heads=4, ent_epochs=3, ent_lr=1e-3,
    # BLT patching
    target_patch_size=4.0,  # entropy threshold chosen so the average patch is ~4 bytes
    max_patch_len=8,        # force a boundary if a patch grows longer than this
    fixed_stride=4,         # BLT-fixed uses this stride -> same #patches on average (fair compute)
    # BLT classifier
    d_local=128, local_layers=2, local_heads=4, local_window=32,
    d_global=256, global_layers=4, global_heads=4,
    hash_ngrams=(3, 4, 5), hash_buckets=20_000,
    # BPE baseline
    bpe_d=256, bpe_layers=4, bpe_heads=4,
    # training
    batch_size=64, epochs=10, lr=3e-4, weight_decay=0.05, dropout=0.2,

)


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

set_seed(CFG["seed"])

device: cuda


In [3]:
from datasets import load_dataset, concatenate_datasets

def load_sms():
    ds = load_dataset("ucirvine/sms_spam", split="train")          # columns: sms, label (0 ham, 1 spam)
    return pd.DataFrame({"text": [s.strip() for s in ds["sms"]],
                         "label": [int(l) for l in ds["label"]],
                         "source": "sms"})

def load_urls():
    dd = load_dataset("pirocheto/phishing-url")                     # splits: train, test
    ds = concatenate_datasets([dd[s] for s in dd.keys()])          # we re-split ourselves after dedup
    raw = ds["status"]
    if isinstance(raw[0], str):
        labels = [1 if r.strip().lower() == "phishing" else 0 for r in raw]
    else:
        labels = [int(r) for r in raw]                              # already 0/1
    return pd.DataFrame({"text": [u.strip() for u in ds["url"]],
                         "label": labels, "source": "url"})

df = pd.concat([load_sms(), load_urls()], ignore_index=True)


df = df.drop_duplicates(subset=["text", "label"])
conflict = df.groupby("text")["label"].nunique()
df = df[~df["text"].isin(conflict[conflict > 1].index)].reset_index(drop=True)


strat = df["source"] + "_" + df["label"].astype(str)
train_df, tmp_df = train_test_split(df, test_size=0.30, stratify=strat, random_state=CFG["seed"])
strat_tmp = tmp_df["source"] + "_" + tmp_df["label"].astype(str)
val_df, test_df = train_test_split(tmp_df, test_size=2/3, stratify=strat_tmp, random_state=CFG["seed"])
train_df, val_df, test_df = [d.reset_index(drop=True) for d in (train_df, val_df, test_df)]

print(df.groupby(["source", "label"]).size().rename("count"))
print(f"train {len(train_df)} | val {len(val_df)} | test {len(test_df)}")

README.md:   0%|          | 0.00/4.98k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  359kB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5574 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

...

data/test.parquet: reconstructing file:   0%|          |  0.00B /  431kB            

data/test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7658 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3772 [00:00<?, ? examples/s]

source  label
sms     0        4518
        1         642
url     0        5715
        1        5714
Name: count, dtype: int64
train 11612 | val 1659 | test 3318


In [4]:
LEET = {"a": "4", "e": "3", "i": "1", "o": "0", "s": "5", "t": "7", "l": "1", "g": "9"}
HOMOGLYPH = {"a": "а", "e": "е", "o": "о", "p": "р", "c": "с", "x": "х", "i": "і", "y": "у"}
NOISE_CHARS = [".", "-", "_", "*"]

def attack_leet(t, rng, p=0.35):
    return "".join(LEET[c.lower()] if c.lower() in LEET and rng.random() < p else c for c in t)

def attack_homoglyph(t, rng, p=0.35):
    return "".join(HOMOGLYPH[c.lower()] if c.lower() in HOMOGLYPH and rng.random() < p else c for c in t)

def attack_noise(t, rng, p=0.10):
    out = []
    for c in t:
        out.append(c)
        if c.isalpha() and rng.random() < p:
            out.append(rng.choice(NOISE_CHARS))
    return "".join(out)

def attack_case(t, rng, p=0.30):
    return "".join(c.swapcase() if c.isalpha() and rng.random() < p else c for c in t)

ATTACKS = {"leet": attack_leet, "homoglyph": attack_homoglyph, "char_noise": attack_noise, "case_flip": attack_case}

rng = random.Random(CFG["seed"])
mal_test = test_df[test_df["label"] == 1].reset_index(drop=True)
attacked_sets = {name: mal_test.assign(text=[fn(t, rng) for t in mal_test["text"]]) for name, fn in ATTACKS.items()}

ex = mal_test["text"].iloc[0]
print("original :", ex[:100])
for name, d in attacked_sets.items():
    print(f"{name:10s}:", d["text"].iloc[0][:100])

original : http://stolizaparketa.ru/wp-content/themes/twentyfifteen/css/read/chinavali/index.php?email=_xxx@yyy
leet      : ht7p://57oliz4park37a.ru/wp-c0n7ent/7heme5/twen7yf1ft33n/c5s/read/chinavali/index.php?emai1=_xxx@yyy
homoglyph : http://stolizаparketа.ru/wp-сontent/thеmes/twentуfіfteen/css/rеаd/chinavali/indeх.рhр?emаil=_xxx@yуу
char_noise: h-ttp://stoli.za_parketa.ru/wp-c*onten.t_/th_emes/twentyfift.een/css/read/ch-inavali/inde.x.p-hp?ema
case_flip : http://STolizApaRKeTa.ru/wP-contENT/themeS/tWeNtyFIfTeeN/cSs/read/chinavalI/indEx.php?EMail=_xxX@yyY


In [5]:
PAD, BOS = 256, 257          # byte ids 0..255, then PAD and BOS

def to_bytes(text):
    return list(text.encode("utf-8"))[: CFG["max_bytes"]] or [32]   # never empty

class CausalByteLM(nn.Module):
    """Tiny autoregressive byte LM. Used ONLY to compute next-byte entropy for patching."""
    def __init__(self, d, layers, heads, max_len):
        super().__init__()
        self.emb = nn.Embedding(258, d)
        self.pos = nn.Embedding(max_len, d)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout=0.1, batch_first=True, norm_first=True)
        self.tf = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.ln = nn.LayerNorm(d)
        self.head = nn.Linear(d, 256)

    def forward(self, x, pad_mask):
        L = x.size(1)
        causal = torch.triu(torch.ones(L, L, dtype=torch.bool, device=x.device), 1)
        h = self.emb(x) + self.pos(torch.arange(L, device=x.device))
        h = self.tf(h, mask=causal, src_key_padding_mask=pad_mask)
        return self.head(self.ln(h))

def lm_batch(byte_lists):
    """input = [BOS] + bytes; logits at position j predict bytes[j]."""
    L = max(len(b) for b in byte_lists) + 1
    x = torch.full((len(byte_lists), L), PAD, dtype=torch.long)
    y = torch.full((len(byte_lists), L), -100, dtype=torch.long)
    for i, b in enumerate(byte_lists):
        x[i, 0] = BOS
        x[i, 1:len(b) + 1] = torch.tensor(b)
        y[i, :len(b)] = torch.tensor(b)
    return x, y, x == PAD

train_bytes = [to_bytes(t) for t in train_df["text"]]
ent_lm = CausalByteLM(CFG["ent_d"], CFG["ent_layers"], CFG["ent_heads"], CFG["max_bytes"] + 1).to(DEVICE)
opt = torch.optim.AdamW(ent_lm.parameters(), lr=CFG["ent_lr"])

for ep in range(CFG["ent_epochs"]):
    ent_lm.train(); idx = np.random.permutation(len(train_bytes)); tot, n = 0.0, 0
    for s in range(0, len(idx), CFG["batch_size"]):
        x, y, m = lm_batch([train_bytes[i] for i in idx[s:s + CFG["batch_size"]]])
        x, y, m = x.to(DEVICE), y.to(DEVICE), m.to(DEVICE)
        loss = F.cross_entropy(ent_lm(x, m).reshape(-1, 256), y.reshape(-1), ignore_index=-100)
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item(); n += 1
    print(f"entropy-LM epoch {ep + 1}: loss {tot / n:.3f} ({tot / n / math.log(2):.2f} bits/byte)")

@torch.no_grad()
def byte_entropies(byte_lists, bs=128):
    """Returns, for each text, an array H[j] = entropy (nats) of the model's prediction of byte j."""
    ent_lm.eval(); out = []
    for s in range(0, len(byte_lists), bs):
        chunk = byte_lists[s:s + bs]
        x, _, m = lm_batch(chunk)
        logp = F.log_softmax(ent_lm(x.to(DEVICE), m.to(DEVICE)).float(), -1)
        H = -(logp.exp() * logp).sum(-1).cpu().numpy()
        out += [H[i, :len(b)] for i, b in enumerate(chunk)]
    return out

entropy-LM epoch 1: loss 3.008 (4.34 bits/byte)
entropy-LM epoch 2: loss 2.605 (3.76 bits/byte)
entropy-LM epoch 3: loss 2.493 (3.60 bits/byte)


In [6]:
train_H = byte_entropies(train_bytes)
all_H = np.concatenate([h[1:] for h in train_H])                 # position 0 always starts a patch
THETA = float(np.quantile(all_H, 1 - 1 / CFG["target_patch_size"]))
print(f"global entropy threshold theta = {THETA:.3f} nats")

def entropy_patch_ids(H):
    ids, pid, run = np.zeros(len(H), dtype=np.int64), -1, 0
    for j, h in enumerate(H):
        if j == 0 or h > THETA or run >= CFG["max_patch_len"]:
            pid += 1; run = 0
        ids[j] = pid; run += 1
    return ids

def fixed_patch_ids(n):
    return np.arange(n, dtype=np.int64) // CFG["fixed_stride"]

def show_patches(text):
    b = to_bytes(text); ids = entropy_patch_ids(byte_entropies([b])[0])
    pieces, cur = [], []
    for byte, pid in zip(b, ids):
        if cur and pid != prev:
            pieces.append(bytes(cur).decode("utf-8", "replace")); cur = []
        cur.append(byte); prev = pid
    pieces.append(bytes(cur).decode("utf-8", "replace"))
    return " | ".join(pieces)

for t in [mal_test["text"].iloc[0], attacked_sets["leet"]["text"].iloc[0]]:
    print(show_patches(t)[:200], "\n")

global entropy threshold theta = 3.027 nats
http://s | tolizapa | rketa.ru | /wp- | content/ | themes/ | twentyfi | fteen/ | css/ | re | ad/ | chinaval | i/ | index.ph | p? | e | mail= | _ | x | x | x | @ | y | y | y | .com_ 

ht7 | p://5 | 7oliz | 4 | p | ark37a.r | u/wp- | c0n7e | nt/ | 7he | me5/ | twe | n7y | f | 1f | t3 | 3n/ | c | 5 | s/ | re | a | d | / | c | hina | va | li/ | inde | x.php? | e | mai1 | = | _ | x | x 



In [7]:
from transformers import AutoTokenizer
bpe_tok = AutoTokenizer.from_pretrained("gpt2")

def make_examples(frame, mode):
    texts, labels = list(frame["text"]), list(frame["label"])
    if mode == "bpe":
        enc = bpe_tok(texts, truncation=True, max_length=CFG["max_bpe_tokens"])["input_ids"]
        return [(e or [bpe_tok.eos_token_id], None, y) for e, y in zip(enc, labels)]
    bl = [to_bytes(t) for t in texts]
    if mode == "blt_entropy":
        pids = [entropy_patch_ids(h) for h in byte_entropies(bl)]
    else:
        pids = [fixed_patch_ids(len(b)) for b in bl]
    return [(b, p, y) for b, p, y in zip(bl, pids, labels)]

def collate(batch, mode):
    L = max(len(b) for b, _, _ in batch)
    pad_val = bpe_tok.eos_token_id if mode == "bpe" else PAD
    x = torch.full((len(batch), L), pad_val, dtype=torch.long)
    pid = torch.full((len(batch), L), -1, dtype=torch.long)
    mask = torch.ones(len(batch), L, dtype=torch.bool)               # True = padding
    for i, (b, p, _) in enumerate(batch):
        x[i, :len(b)] = torch.tensor(b); mask[i, :len(b)] = False
        if p is not None:
            pid[i, :len(p)] = torch.tensor(p)
    y = torch.tensor([lab for _, _, lab in batch])
    return x, pid, mask, y

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [8]:
class HashNgramEmbedding(nn.Module):
    """BLT's hash n-gram embeddings: each byte also gets embeddings of the n-grams ending at it."""
    def __init__(self, d, ngrams, buckets):
        super().__init__()
        self.ngrams, self.buckets = ngrams, buckets
        self.tables = nn.ModuleList([nn.Embedding(buckets, d) for _ in ngrams])

    def forward(self, x):
        out = 0
        for n, table in zip(self.ngrams, self.tables):
            h = torch.zeros_like(x)
            for k in range(n - 1, -1, -1):                            # polynomial hash of bytes i-n+1 .. i
                h = h * 259 + F.pad(x, (k, 0), value=PAD)[:, : x.size(1)]
            out = out + table(h % self.buckets)
        return out

class Block(nn.Module):
    """Pre-norm transformer block. Same math in train and eval (no PyTorch fast path)."""
    def __init__(self, d, heads, dropout):
        super().__init__()
        self.h, self.p = heads, dropout
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv, self.proj = nn.Linear(d, 3 * d), nn.Linear(d, d)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, allowed):                      # allowed: (B,1,L,L) bool, True = may attend
        B, L, D = x.shape
        q, k, v = self.qkv(self.ln1(x)).view(B, L, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed,
                                           dropout_p=self.p if self.training else 0.0)
        x = x + self.drop(self.proj(a.transpose(1, 2).reshape(B, L, D)))
        return x + self.drop(self.ff(self.ln2(x)))

class EncoderStack(nn.Module):
    def __init__(self, d, layers, heads, dropout):
        super().__init__()
        self.blocks = nn.ModuleList([Block(d, heads, dropout) for _ in range(layers)])
        self.ln = nn.LayerNorm(d)

    def forward(self, h, pad_mask, attn_mask=None):
        B, L, _ = h.shape
        blocked = pad_mask[:, None, :].expand(B, L, L)              # can't attend to padding keys
        if attn_mask is not None:
            blocked = blocked | attn_mask[None]                     # e.g. local sliding window
        eye = torch.eye(L, dtype=torch.bool, device=h.device)
        allowed = ((~blocked) | eye)[:, None]                       # always see yourself -> no all-masked rows
        for blk in self.blocks:
            h = blk(h, allowed)
        return self.ln(h)

class PatchCrossAttention(nn.Module):
    """Patch queries (init = mean of their bytes) attend ONLY to the bytes inside their own patch."""
    def __init__(self, d_local, d_global, heads):
        super().__init__()
        self.h, self.dh = heads, d_global // heads
        self.q = nn.Linear(d_local, d_global); self.k = nn.Linear(d_local, d_global)
        self.v = nn.Linear(d_local, d_global); self.o = nn.Linear(d_global, d_global)
        self.ln = nn.LayerNorm(d_global)

    def forward(self, hb, patch_ids):
        B, L, _ = hb.shape
        P = int(patch_ids.max().item()) + 1
        member = patch_ids.unsqueeze(1) == torch.arange(P, device=hb.device).view(1, P, 1)   # (B,P,L)
        counts = member.sum(-1, keepdim=True)                                                 # (B,P,1)
        valid = counts.squeeze(-1) > 0                                                        # (B,P)
        init = (member.float() @ hb) / counts.clamp(min=1)                                    # (B,P,d_local)

        q = self.q(init).view(B, P, self.h, self.dh).transpose(1, 2)
        k = self.k(hb).view(B, L, self.h, self.dh).transpose(1, 2)
        v = self.v(hb).view(B, L, self.h, self.dh).transpose(1, 2)
        allowed = member.clone(); allowed[~valid, 0] = True                  # avoid all-masked rows (NaN)
        scores = (q @ k.transpose(-1, -2)) / math.sqrt(self.dh)
        scores = scores.masked_fill(~allowed.unsqueeze(1), float("-inf"))
        out = (scores.softmax(-1) @ v).transpose(1, 2).reshape(B, P, -1)
        patches = self.ln(self.q(init) + self.o(out))                        # residual
        return patches * valid.unsqueeze(-1), valid

class BLTClassifier(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.byte_emb = nn.Embedding(258, c["d_local"])
        self.pos = nn.Embedding(c["max_bytes"], c["d_local"])
        self.hash_emb = HashNgramEmbedding(c["d_local"], c["hash_ngrams"], c["hash_buckets"])
        self.local = EncoderStack(c["d_local"], c["local_layers"], c["local_heads"], c["dropout"])
        self.pool = PatchCrossAttention(c["d_local"], c["d_global"], c["global_heads"])
        self.patch_pos = nn.Embedding(c["max_bytes"], c["d_global"])
        self.glob = EncoderStack(c["d_global"], c["global_layers"], c["global_heads"], c["dropout"])
        self.head = nn.Linear(c["d_global"], 2)
        self.window = c["local_window"]

    def forward(self, x, patch_ids, pad_mask):
        L = x.size(1)
        pos = torch.arange(L, device=x.device)
        h = self.byte_emb(x) + self.hash_emb(x) + self.pos(pos)
        local_mask = (pos.view(-1, 1) - pos.view(1, -1)).abs() > self.window   # local sliding window
        h = self.local(h, pad_mask, local_mask)
        h = h.masked_fill(pad_mask.unsqueeze(-1), 0.0)      # pad rows can be NaN (no visible keys); zero them
        patches, valid = self.pool(h, patch_ids)
        patches = patches + self.patch_pos(torch.arange(patches.size(1), device=x.device))
        g = self.glob(patches, ~valid)
        pooled = (g * valid.unsqueeze(-1)).sum(1) / valid.sum(1, keepdim=True)
        return self.head(pooled)

class BPEClassifier(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.emb = nn.Embedding(len(bpe_tok), c["bpe_d"])
        self.pos = nn.Embedding(c["max_bpe_tokens"], c["bpe_d"])
        self.enc = EncoderStack(c["bpe_d"], c["bpe_layers"], c["bpe_heads"], c["dropout"])
        self.head = nn.Linear(c["bpe_d"], 2)

    def forward(self, x, patch_ids, pad_mask):
        h = self.emb(x) + self.pos(torch.arange(x.size(1), device=x.device))
        h = self.enc(h, pad_mask)
        keep = (~pad_mask).unsqueeze(-1).float()
        return self.head((h * keep).sum(1) / keep.sum(1))

def count_params(m, exclude_embeddings=False):
    n = sum(p.numel() for p in m.parameters())
    if exclude_embeddings:
        n -= sum(p.numel() for mod in m.modules() if isinstance(mod, nn.Embedding) for p in mod.parameters())
    return n

In [9]:
def batches(examples, mode, shuffle):
    order = np.random.permutation(len(examples)) if shuffle else np.arange(len(examples))
    for s in range(0, len(order), CFG["batch_size"]):
        yield collate([examples[i] for i in order[s:s + CFG["batch_size"]]], mode)


@torch.no_grad()
def predict(model, examples, mode):
    model.eval(); preds = []
    for x, pid, m, _ in batches(examples, mode, shuffle=False):
        logits = model(x.to(DEVICE), pid.to(DEVICE), m.to(DEVICE))
        assert torch.isfinite(logits).all(), f"{mode}: non-finite logits at eval"
        preds += logits.argmax(-1).cpu().tolist()
    return np.array(preds)

def train_model(mode):
    set_seed(CFG["seed"])
    model = (BPEClassifier(CFG) if mode == "bpe" else BLTClassifier(CFG)).to(DEVICE)
    tr, va = make_examples(train_df, mode), make_examples(val_df, mode)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    steps = CFG["epochs"] * math.ceil(len(tr) / CFG["batch_size"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG["lr"], total_steps=steps, pct_start=0.1)
    best_f1, best_state = -1, None
    print(f"\n== {mode} | params {count_params(model)/1e6:.2f}M "
          f"(non-embedding {count_params(model, True)/1e6:.2f}M) ==")
    for ep in range(CFG["epochs"]):
        model.train(); t0, tot, n = time.time(), 0.0, 0
        for x, pid, m, y in batches(tr, mode, shuffle=True):
            loss = F.cross_entropy(model(x.to(DEVICE), pid.to(DEVICE), m.to(DEVICE)), y.to(DEVICE))
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); tot += loss.item(); n += 1
        vf1 = f1_score(val_df["label"], predict(model, va, mode))
        print(f"epoch {ep + 1}: loss {tot / n:.4f} | val F1 {vf1:.4f} | {time.time() - t0:.0f}s")
        if vf1 > best_f1:
            best_f1 = vf1; best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model

def evaluate(model, mode):
    row = {"model": mode}
    te = make_examples(test_df, mode); p = predict(model, te, mode); y = test_df["label"].values
    row["acc"] = accuracy_score(y, p); row["F1"] = f1_score(y, p)
    row["precision"] = precision_score(y, p); row["recall_clean"] = recall_score(y, p)
    for src in ["sms", "url"]:
        s = (test_df["source"] == src).values
        row[f"F1_{src}"] = f1_score(y[s], p[s])
    # robustness: fraction of OBFUSCATED malicious messages that are still caught
    for name, frame in attacked_sets.items():
        row[f"recall_{name}"] = predict(model, make_examples(frame, mode), mode).mean()
    return row

In [10]:
rng = random.Random(42)
attacked_full = {name: test_df.assign(text=[fn(t, rng) for t in test_df["text"]])
                 for name, fn in ATTACKS.items()}

def evaluate(model, mode):
    y = test_df["label"].values
    p_clean = predict(model, make_examples(test_df, mode), mode)
    row = {"model": mode, "F1_clean": f1_score(y, p_clean),
           "F1_sms": f1_score(y[test_df["source"] == "sms"], p_clean[test_df["source"] == "sms"]),
           "F1_url": f1_score(y[test_df["source"] == "url"], p_clean[test_df["source"] == "url"])}
    for name, frame in attacked_full.items():
        p = predict(model, make_examples(frame, mode), mode)
        row[f"F1_{name}"] = f1_score(y, p)                 # missed scams AND false alarms
        row[f"flip_{name}"] = (p != p_clean).mean()        # lower = more robust
    return row

rows, models = [], {}
for seed in [42, 43, 44]:
    CFG["seed"] = seed
    for mode in ["bpe", "blt_fixed", "blt_entropy"]:
        m = train_model(mode)
        r = evaluate(m, mode); r["seed"] = seed; rows.append(r)
        models[mode] = m                                    # keep last seed for cells 12-13
CFG["seed"] = 42

res_all = pd.DataFrame(rows)
summary = res_all.drop(columns="seed").groupby("model").agg(["mean", "std"]).round(4)
flip_cols = [c for c in summary.columns.levels[0] if c.startswith("flip_")]
print("Flip rate under attack (mean ± std over 3 seeds; lower = more robust):")
print(summary[flip_cols])
summary


== bpe | params 16.06M (non-embedding 3.16M) ==
epoch 1: loss 0.4284 | val F1 0.8310 | 4s
epoch 2: loss 0.2435 | val F1 0.8791 | 3s
epoch 3: loss 0.1736 | val F1 0.8991 | 3s
epoch 4: loss 0.1297 | val F1 0.9056 | 3s
epoch 5: loss 0.0855 | val F1 0.9136 | 3s
epoch 6: loss 0.0521 | val F1 0.9117 | 3s
epoch 7: loss 0.0307 | val F1 0.9024 | 3s
epoch 8: loss 0.0205 | val F1 0.9089 | 3s
epoch 9: loss 0.0154 | val F1 0.9072 | 3s
epoch 10: loss 0.0123 | val F1 0.9130 | 3s

== blt_fixed | params 11.53M (non-embedding 3.72M) ==
epoch 1: loss 0.4400 | val F1 0.8305 | 5s
epoch 2: loss 0.2814 | val F1 0.8845 | 5s
epoch 3: loss 0.1949 | val F1 0.8813 | 5s
epoch 4: loss 0.1277 | val F1 0.8967 | 5s
epoch 5: loss 0.0758 | val F1 0.9056 | 5s
epoch 6: loss 0.0418 | val F1 0.9045 | 5s
epoch 7: loss 0.0210 | val F1 0.8998 | 5s
epoch 8: loss 0.0128 | val F1 0.9015 | 5s
epoch 9: loss 0.0079 | val F1 0.9032 | 5s
epoch 10: loss 0.0073 | val F1 0.8996 | 5s

== blt_entropy | params 11.53M (non-embedding 3.72M) 

F1_clean          F1_sms          F1_url         F1_leet          \
                mean     std    mean     std    mean     std    mean     std   
model                                                                          
blt_entropy   0.9058  0.0077  0.9018  0.0108  0.9063  0.0095  0.8013  0.0126   
blt_fixed     0.9021  0.0082  0.9041  0.0235  0.9019  0.0069  0.8172  0.0094   
bpe           0.9091  0.0059  0.8896  0.0079  0.9114  0.0059  0.6703  0.0066   

            flip_leet          ... flip_homoglyph         F1_char_noise  \
                 mean     std  ...           mean     std          mean   
model                          ...                                        
blt_entropy    0.1574  0.0207  ...         0.1539  0.0269        0.8556   
blt_fixed      0.1361  0.0053  ...         0.1149  0.0078        0.8540   
bpe            0.3576  0.0105  ...         0.1771  0.0250        0.8239   

                    flip_char_noise         F1_case_flip          \
                std            mean     std         mean     std   
model                                                              
blt_entropy  0.0137          0.0875  0.0071       0.7615  0.0134   
blt_fixed    0.0093          0.0804  0.0097       0.7919  0.0190   
bpe          0.0106          0.1126  0.0091       0.7178  0.0023   

            flip_case_flip          
                      mean     std  
model                               
blt_entropy         0.1935  0.0139  
blt_fixed           0.1491  0.0125  
bpe                 0.2498  0.0067  

[3 rows x 22 columns]

In [11]:
summary[["F1_clean", "F1_sms", "F1_url"]]

F1_clean          F1_sms          F1_url        
                mean     std    mean     std    mean     std
model                                                       
blt_entropy   0.9058  0.0077  0.9018  0.0108  0.9063  0.0095
blt_fixed     0.9021  0.0082  0.9041  0.0235  0.9019  0.0069
bpe           0.9091  0.0059  0.8896  0.0079  0.9114  0.0059